In [17]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

PROJET = Path("/info/etu/m1/s2603634/Projet/ChallengeMachineLearning")
if str(PROJET) not in sys.path:
    sys.path.insert(0, str(PROJET))

from automl.data import lireDataset

DATA_ROOT = Path("/info/corpus/ChallengeMachineLearning")

In [18]:
data_A = lireDataset(DATA_ROOT / "data_A")
data_A.split()   # découpe en train, valid et test

print("tâche :", data_A.tache)
print("colonnes catégorielles :", data_A.types.count("Categorical"))
print("colonnes numériques :", data_A.types.count("Numerical"))
print("train / valid / test :", data_A.train.x.shape, data_A.valid.x.shape, data_A.test.x.shape)
print("NaN dans le train :", int(np.isnan(data_A.train.x).sum()))

08:43:06 | INFO | Lecture data_A : dense, shape=(34190, 24), solution=(34190, 3) (0.12s)
08:43:06 | INFO | data_A : tâche détectée = multilabel
08:43:06 | INFO | data_A : 6.6 Mo en mémoire
08:43:06 | INFO | Stratification activée (multilabel) : 34190 observations, 8 classes, la plus rare a 173 exemples, la plus fréquente 13777
08:43:06 | INFO | Stratification activée (multilabel) : 27352 observations, 8 classes, la plus rare a 138 exemples, la plus fréquente 11022
08:43:06 | INFO | Split data_A : train=23933, valid=3419, test=6838
tâche : multilabel
colonnes catégorielles : 13
colonnes numériques : 11
train / valid / test : (23933, 24) (3419, 24) (6838, 24)
NaN dans le train : 6365


In [19]:
X = data_A.data   # toutes les données de data_A

print("NaN au total :", int(np.isnan(X).sum()))
print("inf au total :", int(np.isinf(X).sum()))

nan_par_colonne = np.isnan(X).sum(axis=0) # (on n'affiche que les colonnes qui ont des Nan)
for i, nb in enumerate(nan_par_colonne):
    if nb > 0:
        print("colonne", i, "| type :", data_A.types[i], "| NaN :", int(nb))

# vérification utile pour la suite : -1 est-il déjà utilisé comme modalité ?
cols_cat = [i for i, t in enumerate(data_A.types) if t == "Categorical"]
print("-1 dans les colonnes catégorielles :", bool((X[:, cols_cat] == -1).any()))

NaN au total : 9035
inf au total : 0
colonne 0 | type : Categorical | NaN : 1959
colonne 3 | type : Categorical | NaN : 614
colonne 6 | type : Categorical | NaN : 1952
colonne 7 | type : Categorical | NaN : 1941
colonne 13 | type : Categorical | NaN : 590
colonne 21 | type : Categorical | NaN : 1979
-1 dans les colonnes catégorielles : False


In [20]:
def construire_pretraitement(types):
    cols_cat = [i for i, t in enumerate(types) if t == "Categorical"]   # numéros des colonnes catégorielles

    traitement_cat = make_pipeline(
        SimpleImputer(strategy="most_frequent"),      # NaN -> modalité la plus fréquente
        OneHotEncoder(handle_unknown="ignore"),       # une colonne 0/1 par modalité
    )
    return make_column_transformer(
        (traitement_cat, cols_cat),
        remainder="passthrough",                      # les autres colonnes sont gardées telles quelles
        sparse_threshold=0,                           # sortie en tableau normal
    )

In [21]:
pre = construire_pretraitement(data_A.types)
pre.fit(data_A.train.x)    # le nettoyage est appris sur le train uniquement

for nom, x in [("train", data_A.train.x), ("valid", data_A.valid.x), ("test", data_A.test.x)]:
    M = pre.transform(x)
    print(nom, M.shape, "NaN restants :", int(np.isnan(M).sum()))

train (23933, 214) NaN restants : 0
valid (3419, 214) NaN restants : 0
test (6838, 214) NaN restants : 0
